# Lab 7 — From a standard Agent to a Harness Agent

This notebook **imports and constructs** the main Microsoft Agent Framework APIs directly rather than hiding them behind workshop helpers. First build a standard Agent using only Foundry IQ, then add Toolbox tools, Skills, plan / execute mode, todos, and a bounded loop to create a Harness Agent.

**Kernel:** `Select Kernel > Jupyter Kernel... > Python (Foundry Hosted Agent)`. Complete `00-setup.ipynb` in Codespaces or the same local Dev Container first. Kernel sessions are not restored after a container restart; start again by checking connections.

| Stage | Directly constructed components | What to observe |
|---|---|---|
| 1 | `AzureCliCredential` → `FoundryChatClient` → `MCPStreamableHTTPTool` → `Agent(...)` | Minimal client / instructions / tools / run configuration |
| 2 | `FoundryToolbox` + providers → `create_harness_agent(...)` | Task decomposition, lazy Skill loading, Tool Search, multiple tools, and todo completion |

> **Data boundaries and charges:** The notebook runs locally, but models, Foundry IQ, Toolbox, Code Interpreter, and Web Search run on Azure. Use synthetic data only; do not enter secrets, personal information, or customer data. Do not rerun an execution cell while waiting for its result.

## 1. Load connections from Labs 1–4

Read project, model, and Azure AI Search endpoints from `.workshop/context.json`, generated by `00-setup.ipynb`, into Python variables for the following constructors. Foundry IQ and Toolbox use the fixed names created in Labs 3 / 4. No API keys or client secrets are read.

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "scripts" / "configure_workshop.py"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Repository root not found. Open the notebook in your cloned workshop repository."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.lib.workshop_context import load_context, validate_workshop_context  # noqa: E402
from scripts.lib.workshop_runtime import HOSTED_ENVIRONMENT, require_current_runtime  # noqa: E402

require_current_runtime(HOSTED_ENVIRONMENT)
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError("Complete 00-setup.ipynb to create .workshop/context.json.")

context = validate_workshop_context(load_context(context_path))
outputs = context["resource_outputs"]

project_endpoint = outputs["foundry_project_endpoint"]["value"]
model_deployment = outputs["primary_model_deployment_name"]["value"]
search_service_endpoint = outputs["search_service_endpoint"]["value"]
knowledge_base_name = "contoso-travel-knowledge-lab"
toolbox_name = "contoso-travel-toolbox"

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model: {model_deployment}")
print(f"Knowledge base: {knowledge_base_name}")
print(f"Toolbox: {toolbox_name}")

## 2. Construct a standard Agent with Foundry IQ directly

The next 2 cells import the required classes directly from Agent Framework and Azure Identity and assemble them in this order:

```text
AzureCliCredential
  ├─ FoundryChatClient ─┐
  └─ Foundry IQ MCP tool ├─ Agent(...) ─ run(...)
instructions ────────────┘
```

**Middleware** inserts shared behavior before and after processing. Agent Framework provides different types for different targets: this notebook's `ChatMiddleware` wraps `FoundryChatClient` model calls, while `FunctionMiddleware` wraps tool / function execution. Calling `call_next()` inside `process()` advances processing and eventually sends the model request.

The notebook's `request_pacer` spaces model-call starts at least 20 seconds apart to reduce `429 Too Many Requests` errors on the 40K TPM deployment. The standard and Harness Agents share the same instance. It does not pace Foundry IQ or Toolbox calls and is not required by Agent Framework.

In [ ]:
import asyncio
import threading
import time
from collections.abc import Awaitable, Callable
from typing import Any

from agent_framework import (
    Agent,
    ChatContext,
    ChatMiddleware,
    MCPStreamableHTTPTool,
)
from agent_framework.foundry import FoundryChatClient, FoundryToolbox
from azure.identity import AzureCliCredential


class ChatRequestPacer(ChatMiddleware):
    def __init__(self, interval_seconds: float = 20.0) -> None:
        self._interval_seconds = interval_seconds
        self._reservation_lock = threading.Lock()
        self._next_start_at = 0.0

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        del context
        with self._reservation_lock:
            now = time.monotonic()
            start_at = max(now, self._next_start_at)
            self._next_start_at = start_at + self._interval_seconds
        if start_at > now:
            await asyncio.sleep(start_at - now)
        await call_next()


request_pacer = ChatRequestPacer()

In [ ]:
credential = AzureCliCredential()


def search_headers(tool_arguments: dict[str, Any]) -> dict[str, str]:
    del tool_arguments
    access_token = credential.get_token("https://search.azure.com/.default")
    return {"Authorization": f"Bearer {access_token.token}"}


foundry_iq_url = (
    f"{search_service_endpoint.rstrip('/')}/knowledgebases/{knowledge_base_name}"
    "/mcp?api-version=2026-08-01-preview"
)
standard_chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model_deployment,
    credential=credential,
    middleware=[request_pacer],
)
standard_iq_tool = MCPStreamableHTTPTool(
    name="contoso-travel-knowledge",
    url=foundry_iq_url,
    header_provider=search_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
    approval_mode="never_require",
    request_timeout=120,
)

print(type(standard_chat_client).__name__)
print(f"Foundry IQ MCP: {foundry_iq_url}")

In [ ]:
standard_instructions = """
You are Contoso's internal travel and expense assistant.
Use Foundry IQ to research internal policy questions and support your answers with evidence.
Do not guess unverified values. If search finds no information, say it was not found.
This Agent does not yet have cost-calculation or approval-simulation tools.
""".strip()
standard_agent = Agent(
    client=standard_chat_client,
    name="travel_policy_agent",
    description="Answers Contoso travel-policy questions with Foundry IQ.",
    instructions=standard_instructions,
    tools=[standard_iq_tool],
)

standard_question = (
    "I want to book business class on a 12-hour one-way international flight "
    "2 days before departure. "
    "Summarize the last-minute attachment, approvers and their order, application feature name, "
    "and standard maximum business days, with supporting evidence."
)

async with standard_agent:
    standard_response = await standard_agent.run(standard_question)

print(standard_response.text)

Check the answer's citations to both policies. At this point, only Foundry IQ was passed to `Agent(...)` through `tools`. Asking about cost calculations or current information cannot make unavailable tools run.

| Directly constructed component | Role |
|---|---|
| `AzureCliCredential()` | Authenticate as the user signed in through `az login` in the container |
| `FoundryChatClient(...)` | Call the project's model deployment |
| `MCPStreamableHTTPTool(...)` | Expose the Lab 3 knowledge base as an MCP tool |
| `Agent(...)` | Combine client + instructions + tools into a runnable Agent |
| `agent.run(...)` | Execute one request |

## 3. Add Harness execution support to the same configuration

Create a new `FoundryChatClient` and Foundry IQ tool matching the standard Agent, then explicitly add the Lab 4 `FoundryToolbox` and these providers:

- `AgentModeProvider`: plan / execute mode
- `TodoProvider`: todos stored in the session
- `FileSystemAgentFileStore`: file memory limited to `.workshop/harness-memory`
- `toolbox.as_skills_provider(...)`: progressive disclosure of Toolbox Skills
- `todos_remaining(...)`: a bounded loop that repeats only in execute mode with unfinished todos

`create_harness_agent(...)` is the Agent Framework factory that adds these to a standard `Agent`. Disable the Web Search the Harness can add automatically; use it through Lab 4's Toolbox and Tool Search instead.

In [ ]:
harness_instructions = """
You are Contoso's internal travel and expense Harness Agent.

Break complex requests into todos and complete each todo before preparing the final answer.
Do not confuse the roles of information sources and capabilities.
In plan mode, create only the plan and todos. Do not execute Foundry IQ, Skills, Toolbox tools,
or Web Search. Execute them only after the user approves the plan and switches to execute mode.

- Use Foundry IQ to retrieve internal policy and approval-process evidence, and cite it.
- If operational guidance is needed, load the Toolbox Skill and follow its instructions.
- For Toolbox tools, discover via Tool Search's tool_search; execute with call_tool.
- For tool_search query, use one English tool name matching the task, not descriptive prose.
  Use createTripEstimate for estimates, getPerDiem for per diem, createPreapproval for an explicitly
  requested approval simulation, code_interpreter for numerical calculations, and web_search for
  explicitly requested public-information searches. Check the returned official name and inputSchema
  before calling call_tool; do not guess undiscovered tools.
- Use Travel Ops API results for costs, per diem, and approval simulations; do not invent values.
- Use Code Interpreter for ratios and comparisons across multiple results.
- Use Web Search only when the user explicitly requests current public information;
  show retrieval time and sources.

Do not make bookings or grant actual approvals. Run a preapproval simulation only when explicitly
requested by the user, and distinguish it from actual approval. If required information is missing,
list the questions to clarify rather than guessing.
""".strip()

In [ ]:
from contextlib import AsyncExitStack

from agent_framework import (
    AgentModeProvider,
    FileSystemAgentFileStore,
    InMemoryHistoryProvider,
    TodoProvider,
    create_harness_agent,
    get_agent_mode,
    set_agent_mode,
    todos_remaining,
    todos_remaining_message,
)

harness_chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model_deployment,
    credential=credential,
    middleware=[request_pacer],
)
harness_iq_tool = MCPStreamableHTTPTool(
    name="contoso-travel-knowledge",
    url=foundry_iq_url,
    header_provider=search_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
    approval_mode="never_require",
    request_timeout=120,
)
toolbox_url = f"{project_endpoint.rstrip('/')}/toolboxes/{toolbox_name}/mcp?api-version=v1"
toolbox = FoundryToolbox(
    credential,
    name=toolbox_name,
    url=toolbox_url,
    load_tools=True,
    approval_mode="never_require",
)
skills_provider = toolbox.as_skills_provider(
    disable_load_skill_approval=True,
    disable_read_skill_resource_approval=True,
)
todo_provider = TodoProvider()
mode_provider = AgentModeProvider(default_mode="plan")
memory_store = FileSystemAgentFileStore(str(REPO_ROOT / ".workshop" / "harness-memory"))

In [ ]:
harness_agent = create_harness_agent(
    client=harness_chat_client,
    name="travel_harness_agent",
    description=(
        "Plans a travel request, retrieves Contoso policy, loads Toolbox Skills, "
        "and uses Tool Search to select the required travel tools."
    ),
    agent_instructions=harness_instructions,
    tools=[harness_iq_tool, toolbox],
    history_provider=InMemoryHistoryProvider(),
    max_context_window_tokens=128_000,
    max_output_tokens=16_384,
    todo_provider=todo_provider,
    mode_provider=mode_provider,
    file_memory_store=memory_store,
    skills_provider=skills_provider,
    disable_web_search=True,
    disable_tool_auto_approval=True,
    loop_should_continue=todos_remaining(looping_modes=["execute"]),
    loop_next_message=todos_remaining_message,
    loop_max_iterations=6,
)

resource_stack = AsyncExitStack()
await resource_stack.__aenter__()
await resource_stack.enter_async_context(harness_agent)
harness_session = harness_agent.create_session()
print(type(harness_agent).__name__)
print(f"Mode: {get_agent_mode(harness_session)}")

### 3-1. Helper for observing answers and tool calls

`AgentResponse.messages` includes model-requested function/MCP calls and results as well as final text. The following helper lists content types and tool names without rewriting the answer.

In [ ]:
from typing import Any


def observed_actions(response: Any) -> list[str]:
    actions = []
    for message in response.messages:
        for content in message.contents:
            name = getattr(content, "name", None)
            if name:
                actions.append(str(name))
    return actions


def current_todos(session: Any) -> list[dict[str, Any]]:
    state = session.state.get(todo_provider.source_id, {})
    return list(state.get("items", [])) if isinstance(state, dict) else []

## 4. Break a complex request into todos in plan mode

Include policies, APIs, calculations, current information, and an approval simulation in one request. Starting in plan mode, the Harness Agent creates an execution plan and todos, then waits for approval.

In [ ]:
from datetime import date, timedelta

start_date = date.today() + timedelta(days=30)
end_date = start_date + timedelta(days=2)
complex_request = f"""
Assume 1 traveler is going from Tokyo to New York from {start_date.isoformat()} through
{end_date.isoformat()} in business class for a customer workshop, with a budget of JPY 500,000.
Check internal policies and approval procedures with evidence, and summarize the Travel Ops API
cost estimate, difference from budget and budget utilization, 1 current public New York travel
advisory, and a preapproval simulation. Include retrieval time and sources for current information.
Do not make actual bookings or approvals.
""".strip()
print(complex_request)

In [ ]:
plan_response = await harness_agent.run(complex_request, session=harness_session)
print(plan_response.text)
print("\nObserved actions:", observed_actions(plan_response))
print("\nTodos:")
for item in current_todos(harness_session):
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "plan"
assert current_todos(harness_session), "No todos were created. Check the plan output."

## 5. Approve the plan and switch the same session to execute mode

Before continuing, verify that the plan includes no actual bookings, approvals, or unnecessary data transfers. `set_agent_mode` changes only the session's mode; its plan-mode todos and conversation remain in the same session.

In execute mode, the loop continues only while incomplete todos remain. `loop_max_iterations=6` is the safety limit.

In [ ]:
set_agent_mode(
    harness_session,
    "execute",
    source_id=mode_provider.source_id,
    available_modes=mode_provider.available_modes,
)
execute_response = await harness_agent.run(
    "I approve this plan. Execute the todos in order and update completed items.",
    session=harness_session,
)
print(execute_response.text)
execute_actions = observed_actions(execute_response)
print("\nObserved actions:")
for action in execute_actions:
    print(f"- {action}")

todos_after_execute = current_todos(harness_session)
print("\nTodos after execute:")
for item in todos_after_execute:
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "execute"
assert todos_after_execute
assert all(item.get("is_complete") is True for item in todos_after_execute), todos_after_execute

## 6. Inspect why the complex request could be solved

Compare Observed actions with the Portal Trace and distinguish these layers:

| Layer | Expected records | Role |
|---|---|---|
| Harness | `todos_add` / `todos_complete`, mode | Request decomposition and progress tracking |
| Skills | `load_skill`, skill resource read | Lazy loading of only the required operational guidance |
| Tool Search | `tool_search` → `call_tool` | Dynamic discovery and execution of required tool schemas |
| Foundry IQ | `knowledge_base_retrieve` | Evidence for internal policies and approval procedures |
| Toolbox tools | Travel Ops / Code Interpreter / Web Search | API results, calculations, and explicitly requested current information |

Tool Search is not task decomposition. Harness mode / todos / session / loop manage the work; Tool Search discovers tools for each todo. Web Search results vary, so check retrieval time, sources, and explicit failure reporting rather than fixed wording.

In [ ]:
components = {
    "chat client": type(harness_chat_client).__name__,
    "Foundry IQ": type(harness_iq_tool).__name__,
    "Toolbox": type(toolbox).__name__,
    "mode provider": type(mode_provider).__name__,
    "todo provider": type(todo_provider).__name__,
    "memory store": type(memory_store).__name__,
    "Harness Agent": type(harness_agent).__name__,
}
for role, component_type in components.items():
    print(f"{role}: {component_type}")

print("Lab 8 does not reuse this Harness; it compares a sequential workflow of standard Agents.")

## 7. Close notebook connections

Explicitly close Toolbox and Foundry IQ MCP sessions and HTTP clients. This does not delete Azure resources; clean up resources in Lab 9.

In [ ]:
await resource_stack.aclose()
credential.close()
print("Notebook resources closed.")

## Next lab

[Lab 8](../labs/08-hosted-multi-agent.md) does not reuse the Harness, reducing Luna token consumption. Instead, it deploys standard intake / policy / reviewer Agents as a sequential Hosted Agent workflow. Lab 8 does not depend on this notebook's session or output, so experienced participants can skip Lab 7.